In [9]:
import copy
import gc
from importlib import reload
import os
import random
import sys

import numpy as np
from sklearn.preprocessing import RobustScaler, StandardScaler

sys.path.append(os.path.abspath(os.path.join('..')))
import models

reload(models)

if 'utils' in sys.modules:
    del sys.modules['utils']
if 'utils.graph_utils' in sys.modules:
    del sys.modules['utils.graph_utils']

import utils.graph_utils
import utils

import torch
import pandas as pd
import mlflow
import seaborn as sns
import matplotlib.pyplot as plt
from joblib import Parallel, delayed
from torch_geometric.loader import DataLoader
from datetime import datetime
from rdkit import Chem
from rdkit.Chem import AllChem
from rdkit.Chem.Scaffolds import MurckoScaffold
from collections import defaultdict
from models import HybridModel
from utils import mol_to_graph, MLFlowManager, train_hybrid_model, evaluate_hybrid_model, HYB_MAP
from rdkit import RDLogger
RDLogger.DisableLog('rdApp.*') # type: ignore


import logging

logging.basicConfig(
    filename="debug_model.log",
    level=logging.INFO,
    format="%(asctime)s %(levelname)s %(message)s",
    filemode="w"
)

logger = logging.getLogger(__name__)

def scaffold_split(df, smiles_col='canonical_smiles', test_size=0.2, seed=42):
    print("Performing Scaffold Split...")
    scaffolds = {}
    for idx, smiles in df[smiles_col].items():
        mol = Chem.MolFromSmiles(smiles)
        if mol:
            scaffolds[idx] = MurckoScaffold.MurckoScaffoldSmiles(mol=mol)
        else:
            scaffolds[idx] = "invalid"

    scaffold_groups = defaultdict(list)
    for idx, scaffold in scaffolds.items():
        scaffold_groups[scaffold].append(idx)
    
    unique_scaffolds = list(scaffold_groups.keys())
    random.seed(seed)
    random.shuffle(unique_scaffolds)
    
    test_count = int(len(df) * test_size)
    train_idx, test_idx = [], []
    current_test_size = 0
    
    for scaffold in unique_scaffolds:
        group = scaffold_groups[scaffold]
        if current_test_size + len(group) <= test_count:
            test_idx.extend(group)
            current_test_size += len(group)
        else:
            train_idx.extend(group)
            
    return df.loc[train_idx], df.loc[test_idx]


def log_regression_plots(y_true, y_pred, run_name):
    plt.figure(figsize=(8, 6))
    sns.scatterplot(x=y_true, y=y_pred, alpha=0.5)
    plt.plot([min(y_true), max(y_true)], [min(y_true), max(y_true)], '--r', lw=2)
    plt.xlabel("Actual pIC50")
    plt.ylabel("Predicted pIC50")
    plt.title(f"Regression Fit - {run_name}")
    plot_path = "pred_vs_actual.png"
    plt.savefig(plot_path)
    mlflow.log_artifact(plot_path)
    plt.close()

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def train_one_run(params, train_loader, test_loader, device, run_name):
    gc.collect()
    torch.cuda.empty_cache()

    model = HybridModel(
        num_node_features=num_node_features_len,
        num_extra_features=num_extra_features_len,
        hidden_channels=params["hidden_channels"]
    ).to(device)

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=params["lr"]
    )

    best_r2 = float("-inf")
    best_model = None
    best_optimizer = None
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode='max',
        patience=3,
        factor=0.5,
    )

    with mf.start_run(run_name):

        for epoch in range(params["epochs"]):

            loss = train_hybrid_model(
                model,
                train_loader,
                optimizer,
                device,
                epoch
            )

            train_r2, train_mae, _, _ = evaluate_hybrid_model(
                model,
                train_loader,
                device
            )

            mlflow.log_metric("train_r2", train_r2, step=epoch)
            mlflow.log_metric("train_mae", train_mae, step=epoch)

            r2, mae, _, _ = evaluate_hybrid_model(
                model,
                test_loader,
                device
            )
            scheduler.step(r2)

            mlflow.log_metric("train_mse", loss, step=epoch)
            mlflow.log_metric("val_r2", r2, step=epoch)
            mlflow.log_metric("val_mae", mae, step=epoch)

            if r2 > best_r2:
                best_r2 = r2
                best_model = copy.deepcopy(model)
                best_optimizer = copy.deepcopy(optimizer)

        return best_r2, best_model, best_optimizer
    
def process_row(row, g_scaler, a_scaler):
    scaled_features = g_scaler.transform(row[features].values.reshape(1, -1)).flatten()
    return mol_to_graph(row['canonical_smiles'], row['pic50'], scaled_features, a_scaler)

seed_everything(42)

features = [
    'alogp', 'psa', 'hba', 'hbd', 'num_ro5_violations', 'qed_weighted',
    'logP_over_PSA', 'HBA_HBD_sum'
]
num_extra_features_len = len(features)
num_node_features_len = 10
parquet_path = "parquets/subset_CHEMBL203_stratified.parquet"
df = pd.read_parquet(parquet_path)
dataset_path = "/home/pkuszn/repos/WSzI/src/notebooks/data/chembl_dataset.pt"
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

train_df, test_df = scaffold_split(df, smiles_col='canonical_smiles', test_size=0.2, seed=42)

scaler = StandardScaler()
scaler.fit(train_df[features])

all_atoms = []
for smiles in train_df['canonical_smiles']:
    mol = Chem.MolFromSmiles(smiles)
    if mol:
        AllChem.ComputeGasteigerCharges(mol) # type: ignore 
        for atom in mol.GetAtoms():
            all_atoms.append([
                float(atom.GetAtomicNum()), 
                float(atom.GetDegree()), 
                float(atom.GetFormalCharge()),
                1.0 if atom.IsInRing() else 0.0, 
                float(atom.GetIsAromatic()),
                float(atom.GetTotalNumHs()), 
                float(atom.GetTotalValence()),
                float(atom.GetMass()), 
                float(HYB_MAP.get(str(atom.GetHybridization()), 0)),
                float(atom.GetProp('_GasteigerCharge'))
            ])

atom_scaler = RobustScaler()
atom_scaler.fit(np.array(all_atoms))

print("Processing Train...")
train_data = Parallel(n_jobs=-1)(delayed(process_row)(row, scaler, atom_scaler) for _, row in train_df.iterrows())
train_data = [d for d in train_data if d is not None]

print("Processing Test...")
test_data = Parallel(n_jobs=-1)(delayed(process_row)(row, scaler, atom_scaler) for _, row in test_df.iterrows())
test_data = [d for d in test_data if d is not None]

train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
test_loader = DataLoader(test_data, batch_size=32, shuffle=False)

batch = next(iter(train_loader))

param_grid = [
    {"lr": 1e-2, "hidden_channels": 32, "epochs": 50},
    {"lr": 5e-3, "hidden_channels": 64, "epochs": 50},
    {"lr": 1e-3, "hidden_channels": 64, "epochs": 100},
    {"lr": 1e-3, "hidden_channels": 128, "epochs": 100},
    {"lr": 5e-4, "hidden_channels": 128, "epochs": 100},
]

mf = MLFlowManager(experiment_name="ChEMBL_HybridModel_Scaffold_Split")

now = str(int(datetime.now().timestamp()))
run_name = f"Hybrid_Run_{now}"
print("Starting Training...")

best_config = None
best_score = float("-inf")
best_model = None
best_optimizer = None
best_runname = None
for i, params in enumerate(param_grid):
    gc.collect()
    run_name = f"Hybrid_tune_{i}_{int(datetime.now().timestamp())}"

    print(f"\nRunning config {i+1}/{len(param_grid)}: {params}")

    score, model, optimizer = train_one_run(
        params,
        train_loader,
        test_loader,
        device,
        run_name
    )

    for k, v in params.items(): # type: ignore
        mlflow.log_param(f"{run_name}-{k}", v)

    if score > best_score:
        best_score = score
        best_config = params
        best_model = model
        best_optimizer = optimizer
        best_runname = run_name
    gc.collect()
    torch.cuda.empty_cache()

mlflow.log_metrics({"best_r2": best_score})
print("\nBEST CONFIG:", best_config)
print("BEST R2:", best_score)
for k, v in best_config.items(): # type: ignore
    mlflow.log_param(f"best_{k}", v)

model_save_path = f"model_{best_runname}_weights.pth"
checkpoint = {
    "model_state_dict": best_model.state_dict(), # type: ignore
    "optimizer_state_dict": best_optimizer.state_dict(), # type: ignore
    "params": best_config,
    "timestamp": datetime.now().isoformat(),
    "global_scaler": scaler,
    "atom_scaler": atom_scaler,
}
torch.save(checkpoint, model_save_path)
print(f"Saved weight to {model_save_path}")
mlflow.log_artifact(model_save_path)

del optimizer
del model

Performing Scaffold Split...
Processing Train...


/home/pkuszn/repos/WSzI/src/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/home/pkuszn/repos/WSzI/src/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/home/pkuszn/repos/WSzI/src/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/home/pkuszn/repos/WSzI/src/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/home/pkuszn/repos/WSzI/src/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, b

Processing Test...


/home/pkuszn/repos/WSzI/src/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/home/pkuszn/repos/WSzI/src/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/home/pkuszn/repos/WSzI/src/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/home/pkuszn/repos/WSzI/src/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/home/pkuszn/repos/WSzI/src/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, b

Starting Training...

Running config 1/5: {'lr': 0.01, 'hidden_channels': 32, 'epochs': 50}
Epoch 0 | TRAIN_MSE=3.5135 | TRAIN_MAE=1.4540 | TRAIN_R2=-0.6025
Epoch 1 | TRAIN_MSE=2.4427 | TRAIN_MAE=1.2577 | TRAIN_R2=-0.1141
Epoch 2 | TRAIN_MSE=2.3264 | TRAIN_MAE=1.2333 | TRAIN_R2=-0.0611
Epoch 3 | TRAIN_MSE=2.1751 | TRAIN_MAE=1.1909 | TRAIN_R2=0.0080
Epoch 4 | TRAIN_MSE=1.9636 | TRAIN_MAE=1.1292 | TRAIN_R2=0.1044
Epoch 5 | TRAIN_MSE=1.8399 | TRAIN_MAE=1.0910 | TRAIN_R2=0.1609
Epoch 6 | TRAIN_MSE=1.6819 | TRAIN_MAE=1.0405 | TRAIN_R2=0.2329
Epoch 7 | TRAIN_MSE=1.5927 | TRAIN_MAE=1.0144 | TRAIN_R2=0.2736
Epoch 8 | TRAIN_MSE=1.5110 | TRAIN_MAE=0.9829 | TRAIN_R2=0.3108
Epoch 9 | TRAIN_MSE=1.4611 | TRAIN_MAE=0.9655 | TRAIN_R2=0.3336
Epoch 10 | TRAIN_MSE=1.3976 | TRAIN_MAE=0.9437 | TRAIN_R2=0.3626
Epoch 11 | TRAIN_MSE=1.3850 | TRAIN_MAE=0.9350 | TRAIN_R2=0.3683
Epoch 12 | TRAIN_MSE=1.3442 | TRAIN_MAE=0.9223 | TRAIN_R2=0.3869
Epoch 13 | TRAIN_MSE=1.3362 | TRAIN_MAE=0.9157 | TRAIN_R2=0.3906
Epoch

MlflowException: INVALID_PARAMETER_VALUE: Changing param values is not allowed. Param with key='best_lr' was already logged with value='0.01' for run ID='20d338e0e50c4bc2906fd74a7f83da34'. Attempted logging new value '0.001'.

The cause of this error is typically due to repeated calls
to an individual run_id event logging.

Incorrect Example:
---------------------------------------
with mlflow.start_run():
    mlflow.log_param("depth", 3)
    mlflow.log_param("depth", 5)
---------------------------------------

Which will throw an MlflowException for overwriting a
logged parameter.

Correct Example:
---------------------------------------
with mlflow.start_run():
    with mlflow.start_run(nested=True):
        mlflow.log_param("depth", 3)
    with mlflow.start_run(nested=True):
        mlflow.log_param("depth", 5)
---------------------------------------

Which will create a new nested run for each individual
model and prevent parameter key collisions within the
tracking store.

In [ ]:
if 'best_model' in globals() and best_model is not None:
    print("Model znaleziony w pamięci! Zapisuję...")
    
    # Ręczny zapis checkpointu
    model_save_path = f"model_RECOVERED_{best_runname}_weights.pth"
    checkpoint = {
        "model_state_dict": best_model.state_dict(),
        "optimizer_state_dict": best_optimizer.state_dict(),
        "params": best_config,
        "timestamp": datetime.now().isoformat(),
        "global_scaler": scaler,
        "atom_scaler": atom_scaler,
    }
    torch.save(checkpoint, model_save_path)
    print(f"Sukces! Model zapisany jako: {model_save_path}")
else:
    print("Błąd: best_model nie istnieje w pamięci. Musisz uruchomić trening ponownie.")

Model znaleziony w pamięci! Zapisuję...
Sukces! Model zapisany jako: model_RECOVERED_Hybrid_tune_3_1781198740_weights.pth
